# Formal chicken-heart daily interpolation

This notebook reruns the official `chicken_heart_analysis` piecewise split-SDE interpolation using the formal training result, while replacing the original half-step interpolation grid with daily intermediate slices:

- `0 -> 1`: add `1/3`, `2/3` for `D5`, `D6`
- `1 -> 2`: add `4/3`, `5/3` for `D8`, `D9`
- `2 -> 3`: add `2.25`, `2.5`, `2.75` for `D11`, `D12`, `D13`

Interpolation logic remains the formal workflow default: interval-local forward simulation starting from the previous observed anchor.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch
import yaml

import os
REPO_ROOT = Path(os.environ.get("CYTOBRIDGE_SOURCE_DIR", ".")).resolve()
PROJECT_DIR = Path(os.environ.get("CYTOBRIDGE_PROJECT_DIR", ".")).resolve()
DATA_DIR = PROJECT_DIR / "data" / "chicken_heart"
sys.path.insert(0, str(REPO_ROOT / "reproduction" / "chicken_heart"))
ANALYSIS_ROOT = Path(os.environ.get("CYTOBRIDGE_HEART_OUTPUT_DIR", PROJECT_DIR / "outputs" / "chicken_heart_paper")).resolve()
CELLTYPE_SHARE_ROOT = DATA_DIR
import CytoBridge as cb
PACKAGE_ROOT = Path(cb.__file__).resolve().parents[1]

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import CytoBridge as cb

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Repo root: {REPO_ROOT}")

In [ ]:
WORKFLOW_CONFIG_PATH = DATA_DIR / "workflow.json"
ALIGNED_H5AD_PATH = DATA_DIR / "aligned.h5ad"
MODEL_DIR = DATA_DIR / "model"
EDGE_PREDICTOR_PATH = DATA_DIR / "edge_classifier" / "chicken_heart_edge_model.pt"
OUTPUT_DIR = ANALYSIS_ROOT / "new_runs_formal_trained" / "formal_daily_piecewise_interpolation_celltypecorrected"
SLICE_DIR = OUTPUT_DIR / "slice_data"
COMM_DIR = OUTPUT_DIR / "communication_slice_data"
CLASSIFIER_CACHE_PATH = DATA_DIR / "classifier_cache" / "classifier_resmlp_432f09f20ff65c0d.pt"
if not CLASSIFIER_CACHE_PATH.is_file():
    raise FileNotFoundError(CLASSIFIER_CACHE_PATH)

for path in (OUTPUT_DIR, SLICE_DIR, COMM_DIR):
    path.mkdir(parents=True, exist_ok=True)

with WORKFLOW_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    workflow_config = json.load(handle)

annotation_key = workflow_config["dataset"]["annotation_key"]
time_key = workflow_config["dataset"]["time_key"]
obsm_key = workflow_config["dataset"]["obsm_key"]
spatial_key = workflow_config["dataset"]["spatial_key"]
concat_spatial = bool(workflow_config["dataset"].get("concat_spatial", True))

observed_times = [float(value) for value in workflow_config["downstream"]["observed"]]
interp_time_points = [
    1.0 / 3.0,
    2.0 / 3.0,
    4.0 / 3.0,
    5.0 / 3.0,
    2.25,
    2.50,
    2.75,
]
requested_plot_points = sorted(observed_times + interp_time_points)

# The formal workflow requires every requested split-SDE output time to lie
# on the fixed split-event grid. The daily points here mix thirds and quarters,
# so we use 1/12 as the shared output-event spacing.
formal_split_sde_dt = float(workflow_config["downstream"]["split_sde_dt"])
formal_split_resample_dt = float(workflow_config["downstream"]["split_resample_dt"])
daily_split_output_grid_dt = 1.0 / 12.0

time_label_map = {
    0.0: "D4",
    round(1.0 / 3.0, 9): "D5",
    round(2.0 / 3.0, 9): "D6",
    1.0: "D7",
    round(4.0 / 3.0, 9): "D8",
    round(5.0 / 3.0, 9): "D9",
    2.0: "D10",
    2.25: "D11",
    2.50: "D12",
    2.75: "D13",
    3.0: "D14",
}

print("Observed times:", observed_times)
print("Interpolated times:", interp_time_points)
print("Formal split_sde_dt:", formal_split_sde_dt)
print("Formal split_resample_dt:", formal_split_resample_dt)
print("Daily split output grid dt:", daily_split_output_grid_dt)
print("Output dir:", OUTPUT_DIR)
print("Corrected package root:", PACKAGE_ROOT)
print("Corrected classifier cache:", CLASSIFIER_CACHE_PATH)

In [ ]:
adata = ad.read_h5ad(ALIGNED_H5AD_PATH)
if annotation_key not in adata.obs.columns:
    raise KeyError(
        f"Expected annotation column '{annotation_key}' in adata.obs, "
        f"but only found: {list(adata.obs.columns)}"
    )

df, resolved_time_key = cb.tl.adata_to_aligned_dataframe(
    adata,
    time_key=time_key,
    obsm_key=obsm_key,
    spatial_key=spatial_key,
    concat_spatial=concat_spatial,
    annotation_key=annotation_key,
)
feature_cols = cb.tl.infer_feature_columns(df, annotation_column=annotation_key)
dim = len(feature_cols)

loaded = cb.tl.load_dynamical_model_from_dir(
    MODEL_DIR,
    dim=dim,
    device=DEVICE,
    edge_predictor_path=EDGE_PREDICTOR_PATH,
)
runtime = cb.tl.build_dynamical_runtime(loaded)

print(f"Resolved time key: {resolved_time_key}")
print(f"Feature dimension: {dim}")
print(f"Loaded weight stage: {loaded.weight_stage}")
print(f"Loaded score stage: {loaded.score_stage}")
display(df.head())

In [ ]:
downstream_cfg = workflow_config["downstream"]
scientific_cfg = workflow_config["scientific"]

result = cb.tl.run_interpolation_workflow(
    df=df,
    dim=dim,
    annotation_key=annotation_key,
    runtime=runtime,
    device=DEVICE,
    output_dir=str(OUTPUT_DIR),
    requested_plot_points=requested_plot_points,
    interp_time_points=interp_time_points,
    max_observed_timepoints=len(observed_times),
    use_real_for_observed=True,
    classifier_cache_path=str(CLASSIFIER_CACHE_PATH),
    classifier_cache_dir=str(CLASSIFIER_CACHE_PATH.parent),
    classifier_adata=adata,
    classifier_time_key=resolved_time_key,
    classifier_obsm_key=obsm_key,
    classifier_spatial_key=spatial_key,
    classifier_concat_spatial=concat_spatial,
    classifier_epochs=int(downstream_cfg["classifier_epochs"]),
    classifier_hidden_size=int(downstream_cfg["classifier_hidden_size"]),
    classifier_lr=float(downstream_cfg["classifier_lr"]),
    classifier_best_metric=str(downstream_cfg["classifier_best_metric"]),
    classifier_strict_stratification=bool(downstream_cfg["classifier_strict_stratification"]),
    classifier_knn_neighbors=int(scientific_cfg["classifier_k"]),
    sde_n_samples=int(downstream_cfg["sde_n_samples"]),
    sde_dt=float(downstream_cfg["sde_dt"]),
    split_sde_dt=formal_split_sde_dt,
    split_sigma_scalar=float(downstream_cfg["split_sigma"]),
    split_daughter_noise_std=float(downstream_cfg["split_daughter_noise_std"]),
    split_growth_alpha=float(downstream_cfg["split_growth_alpha"]),
    split_interaction_m=1024,
    split_resample_dt=daily_split_output_grid_dt,
    split_max_particles=int(downstream_cfg["split_max_particles"]),
    split_sde_piecewise=bool(downstream_cfg["split_sde_piecewise"]),
    split_sde_piecewise_include_end=bool(downstream_cfg["split_sde_piecewise_include_end"]),
    piecewise_observed_sample_mode=str(downstream_cfg["piecewise_observed_sample_mode"]),
    spatial_warp_to_observed=False,
    spatial_warp_to_observed_piecewise=False,
    spatial_warp_visualization_only=False,
    random_seed=int(scientific_cfg["seed"]),
)

print("Simulation time grid:", result.ts_points)
print("Observed time points:", result.observed_time_points)
print("Interpolated time points:", result.interp_points)

In [ ]:
def normalize_time_value(value: float) -> float:
    return round(float(value), 9)


def time_to_key(value: float) -> str:
    return str(float(value))


def time_to_slug(value: float) -> str:
    text = f"{float(value):.9f}".rstrip("0").rstrip(".")
    return text.replace("-", "neg").replace(".", "p")


if "result" not in globals():
    raise RuntimeError("Interpolation result is unavailable because the previous simulation cell failed.")

combined_slices = []
combined_comm_slices = []
slice_manifest = []

for time_value in result.ts_points:
    key = time_to_key(time_value)
    slug = time_to_slug(time_value)
    norm_t = normalize_time_value(time_value)
    time_label = time_label_map.get(norm_t, f"t={norm_t:g}")
    is_observed = any(np.isclose(time_value, obs, atol=1e-9, rtol=0.0) for obs in observed_times)

    adata_slice = result.adata_dict[key].copy()
    adata_slice.obs["time_float"] = float(time_value)
    adata_slice.obs["time_label"] = time_label
    adata_slice.obs["is_observed"] = bool(is_observed)
    adata_slice.uns["time_float"] = float(time_value)
    adata_slice.uns["time_label"] = time_label
    adata_slice.uns["is_observed"] = bool(is_observed)
    adata_slice.uns["workflow"] = "formal_piecewise_daily_interpolation"
    slice_path = SLICE_DIR / f"time_{slug}.h5ad"
    adata_slice.write_h5ad(slice_path)

    comm_slice = result.communication_adata_dict[key].copy()
    comm_slice.obs["time_float"] = float(time_value)
    comm_slice.obs["time_label"] = time_label
    comm_slice.obs["is_observed"] = bool(is_observed)
    comm_slice.uns["time_float"] = float(time_value)
    comm_slice.uns["time_label"] = time_label
    comm_slice.uns["is_observed"] = bool(is_observed)
    comm_slice.uns["workflow"] = "formal_piecewise_daily_interpolation"
    comm_path = COMM_DIR / f"time_{slug}.h5ad"
    comm_slice.write_h5ad(comm_path)

    combined_adata_slice = adata_slice.copy()
    combined_adata_slice.obs_names = [f"{time_label}_{idx}" for idx in range(combined_adata_slice.n_obs)]
    combined_slices.append(combined_adata_slice)

    combined_comm_slice = comm_slice.copy()
    combined_comm_slice.obs_names = [f"{time_label}_{idx}" for idx in range(combined_comm_slice.n_obs)]
    combined_comm_slices.append(combined_comm_slice)

    slice_manifest.append({
        "time_float": float(time_value),
        "time_key": key,
        "time_label": time_label,
        "slug": slug,
        "is_observed": bool(is_observed),
        "slice_origin": adata_slice.uns.get("slice_origin"),
        "source_anchor_time": float(adata_slice.uns.get("source_anchor_time")),
        "n_cells": int(adata_slice.n_obs),
        "slice_h5ad": str(slice_path),
        "communication_h5ad": str(comm_path),
    })

combined_adata = ad.concat(combined_slices, join="outer", merge="same", label="time_key", keys=[time_to_key(t) for t in result.ts_points], index_unique=None)
combined_comm_adata = ad.concat(combined_comm_slices, join="outer", merge="same", label="time_key", keys=[time_to_key(t) for t in result.ts_points], index_unique=None)

combined_adata.uns["time_label_map"] = {str(k): v for k, v in time_label_map.items()}
combined_adata.uns["observed_times"] = observed_times
combined_adata.uns["interp_time_points"] = interp_time_points
combined_adata.uns["workflow"] = "formal_piecewise_daily_interpolation"
combined_comm_adata.uns["time_label_map"] = {str(k): v for k, v in time_label_map.items()}
combined_comm_adata.uns["observed_times"] = observed_times
combined_comm_adata.uns["interp_time_points"] = interp_time_points
combined_comm_adata.uns["workflow"] = "formal_piecewise_daily_interpolation"

combined_path = OUTPUT_DIR / "combined_interpolated_slices.h5ad"
combined_comm_path = OUTPUT_DIR / "combined_interpolated_communication_slices.h5ad"
combined_adata.write_h5ad(combined_path)
combined_comm_adata.write_h5ad(combined_comm_path)

if result.sde_points_split is not None:
    np.save(OUTPUT_DIR / "split_population_trajectory.npy", result.sde_points_split, allow_pickle=True)
if result.slice_labels_split is not None:
    np.save(OUTPUT_DIR / "split_population_labels.npy", np.asarray(result.slice_labels_split, dtype=object), allow_pickle=True)

manifest = {
    "workflow": "formal_piecewise_daily_interpolation",
    "source_system": "chicken_heart_analysis/chicken_heart_ot_retrained_20260823_c72e592",
    "celltype_corrected_share_root": str(CELLTYPE_SHARE_ROOT),
    "package_root": str(PACKAGE_ROOT),
    "aligned_h5ad": str(ALIGNED_H5AD_PATH),
    "model_dir": str(MODEL_DIR),
    "edge_predictor_path": str(EDGE_PREDICTOR_PATH),
    "device": DEVICE,
    "annotation_key": annotation_key,
    "resolved_time_key": resolved_time_key,
    "obsm_key": obsm_key,
    "spatial_key": spatial_key,
    "concat_spatial": concat_spatial,
    "feature_dim": dim,
    "observed_times": observed_times,
    "interp_time_points": interp_time_points,
    "requested_plot_points": requested_plot_points,
    "time_label_map": {str(k): v for k, v in time_label_map.items()},
    "formal_downstream_parameters": {
        "classifier_epochs": int(downstream_cfg["classifier_epochs"]),
        "classifier_hidden_size": int(downstream_cfg["classifier_hidden_size"]),
        "classifier_lr": float(downstream_cfg["classifier_lr"]),
        "classifier_best_metric": str(downstream_cfg["classifier_best_metric"]),
        "classifier_strict_stratification": bool(downstream_cfg["classifier_strict_stratification"]),
        "classifier_knn_neighbors": int(scientific_cfg["classifier_k"]),
        "sde_n_samples": int(downstream_cfg["sde_n_samples"]),
        "sde_dt": float(downstream_cfg["sde_dt"]),
        "split_sde_dt": formal_split_sde_dt,
        "split_sigma_scalar": float(downstream_cfg["split_sigma"]),
        "split_daughter_noise_std": float(downstream_cfg["split_daughter_noise_std"]),
        "split_growth_alpha": float(downstream_cfg["split_growth_alpha"]),
        "split_interaction_m": 1024,
        "split_resample_dt": daily_split_output_grid_dt,
        "formal_split_resample_dt": formal_split_resample_dt,
        "split_max_particles": int(downstream_cfg["split_max_particles"]),
        "split_sde_piecewise": bool(downstream_cfg["split_sde_piecewise"]),
        "split_sde_piecewise_include_end": bool(downstream_cfg["split_sde_piecewise_include_end"]),
        "piecewise_observed_sample_mode": str(downstream_cfg["piecewise_observed_sample_mode"]),
        "random_seed": int(scientific_cfg["seed"]),
    },
    "loaded_weight_stage": loaded.weight_stage,
    "loaded_weight_path": str(loaded.weight_path),
    "loaded_score_stage": loaded.score_stage,
    "loaded_score_path": str(loaded.score_path) if loaded.score_path is not None else None,
    "classifier_cache_path": result.classifier_cache_path,
    "classifier_accuracy": result.classifier_accuracy,
    "classifier_balanced_accuracy": result.classifier_balanced_accuracy,
    "simulation_seeds": result.simulation_seeds,
    "combined_h5ad": str(combined_path),
    "combined_communication_h5ad": str(combined_comm_path),
    "slices": slice_manifest,
}

manifest_path = OUTPUT_DIR / "manifest.json"
with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

pd.DataFrame(slice_manifest)

In [ ]:
summary = pd.DataFrame(slice_manifest)[[
    "time_float",
    "time_label",
    "is_observed",
    "slice_origin",
    "source_anchor_time",
    "n_cells",
]]
display(summary)

print("Saved manifest:", manifest_path)
print("Saved combined h5ad:", combined_path)
print("Saved combined communication h5ad:", combined_comm_path)